In [1]:
import time
from pathlib import Path

CODE = Path.cwd()
ROOT = CODE.parent
DATA = ROOT / 'data'               

import numpy as np
import pandas as pd

from mg_model import modular_MG_model as mod
from specifications import SPEC
from architechture import AC_Arch, DC_Arch, Custom_Arch1

import matplotlib.pyplot as plt
from mg_model import plot_functions as pf

In [2]:
# Load Data 
spot_df = pd.read_csv(DATA / 'spot_df.csv', parse_dates = ['HourUTC'])
price_15min = spot_df.set_index('HourUTC')['SpotPriceEUR'].resample('15min').ffill()

pv_raw = pd.read_csv(DATA / 'pv_raw.csv', index_col = 'time', parse_dates = ['time'])
pv_15min = (pv_raw['P'] / 1e6).resample('15min').interpolate('linear')

Change date to change price and pv profile. Load profile - add it manually here constant load 2

In [3]:
mgmodel = Custom_Arch1

# Day-ahead scheduling test one day
date = '2025-06-01'
price_day = price_15min[date].values
pv_day    = pv_15min[date].values
load_day  = np.full(SPEC.N_T, SPEC.LOAD_MW) 
idx        = price_15min[date].index

oneday = mod.Scenario(n_t = SPEC.N_T, dt = SPEC.DT, data = {'grid': {'price': price_day},
                                            'PV'  : {'production': pv_day},
                                            'bess': {'soc_init': 0.5, 'soc_final': 0.5},
                                            'load': {'demand': load_day}})


t0 = time.perf_counter()
result = mod.build_and_solve(mgmodel, oneday, verbose = False)
runtime = time.perf_counter() - t0   # build + solve + extract, seconds
print(f'model run time {runtime:.3f} s')

def tree(d, pre = ''):
    for k, v in d.items():
        if isinstance(v, dict):
            print(f'{pre}{k}/')
            tree(v, pre + '  ')
        else:
            a = np.asarray(v)
            what = f'array{a.shape}  {a.min():.4g} .. {a.max():.4g}' if a.ndim else f'{v}'
            print(f'{pre}{k:<14s} {what}')

tree(result)          # or tree(c) for just the components


# Results
c = result['components']

print(f"\n{date}   {result['status']}   cost {result['cost_eur']:.2f} EUR   " f"efficiency {result['efficiency'] * 100:.2f} %")
print(f"grid {result['grid_import_MWh']:.3f} MWh   pv {result['DER_generation_MWh']:.3f} MWh   " f"load {result['demand_supply_MWh']:.3f} MWh   losses {result['total_loss_MWh']:.3f} MWh")

print('\nlosses by stage [MWh]')
for k, v in sorted(result['losses_MWh'].items(), key = lambda kv: -kv[1]):
    print(f'  {k:24s} {v:7.4f}   {v / result["total_loss_MWh"] * 100:5.1f} %')

soc = c['bess']['SOC']
cycled = c['bess']['P_discharge'].sum() * SPEC.DT
print(f"\nbess   soc {soc.min():.2f} - {soc.max():.2f} MWh   " f"in {c['bess']['P_charge'].sum() * SPEC.DT:.3f} MWh   out {cycled:.3f} MWh   " f"{cycled / SPEC.BESS_CAP_MWH:.2f} cycles")
print(f"pv     used {c['PV']['P_terminal'].sum() * SPEC.DT:.3f} MWh   " f"curtailed {c['PV']['curtailed'].sum() * SPEC.DT:.3f} MWh")


dispatch = pd.DataFrame({'price': price_day, 'grid': c['grid']['P_terminal'], 'pv': c['PV']['P_terminal'], 'charge': c['bess']['P_charge'], 'discharge': c['bess']['P_discharge'], 'soc': soc[:-1]}, index = idx)
hourly = dispatch.resample('h').agg({'price': 'mean', 'grid': 'mean', 'pv': 'mean', 'charge': 'mean', 'discharge': 'mean', 'soc': 'last'})
print('\nhourly dispatch [EUR/MWh, MW, MWh]')
print(hourly.round(3).to_string())

bess = result['components']['bess']

df_bess = pd.DataFrame({
    'P_charge': pd.Series(bess['P_charge']),
    'P_discharge': pd.Series(bess['P_discharge']),
    'SOC': pd.Series(bess['SOC'])
})


Set parameter Username
Set parameter LicenseID to value 2817640
Academic license - for non-commercial use only - expires 2027-05-04
model run time 2.549 s
modelname      microgrid
date           None
dt             0.25
n_t            96
status         optimal
cost_eur       -76.60897309722058
components/
  grid/
    P_terminal     array(96,)  0 .. 8.284
    P_bus          array(96,)  0 .. 8.16
    losses/
      trafo          array(96,)  0 .. 0.1241
  PV/
    P_terminal     array(96,)  0 .. 1.377
    P_bus          array(96,)  0 .. 1.316
    losses/
      pv_dcdc        array(96,)  0 .. 0.03033
      pv_dcac        array(96,)  0 .. 0.03091
    curtailed      array(96,)  0 .. 0
  bess/
    capacity_MWh   9.0
    throughput_MWh 14.632784327366322
    cycles         1.6258649252629247
    P_charge       array(96,)  0 .. 5
    P_discharge    array(96,)  0 .. 3.09
    SOC            array(97,)  0.9 .. 8.1
    losses/
      bess_dcac_ch   array(96,)  0 .. 0.15
      bess_dcdc_ch   array(96,

In [7]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

col = {'grid': '#2a78d6', 'pv': '#eb6834', 'charge': '#1baf7a', 'discharge': '#eda100', 'load': '#e87ba4'}
ink = '#52514e'

b = c['bess']
fig = make_subplots(rows = 3, cols = 1, shared_xaxes = True, vertical_spacing = 0.05, row_heights = [0.5, 0.25, 0.25])

for nm, y in [('grid', c['grid']['P_terminal']), ('pv', c['PV']['P_terminal']), ('charge', b['P_charge']),
              ('discharge', b['P_discharge']), ('load', c['load']['P_terminal'])]:
    fig.add_trace(go.Scatter(x = idx, y = y, name = nm, line = dict(color = col[nm], width = 2)), row = 1, col = 1)

fig.add_trace(go.Scatter(x = idx, y = pv_day, name = 'pv available', line = dict(color = col['pv'], width = 1.5, dash = 'dot')), row = 1, col = 1)   #gap to pv = curtailment

fig.add_trace(go.Scatter(x = idx, y = soc[:SPEC.N_T], name = 'soc', line = dict(color = ink, width = 2), showlegend = False), row = 2, col = 1)
fig.add_trace(go.Scatter(x = idx, y = price_day, name = 'price', line = dict(color = ink, width = 2, shape = 'hv'), showlegend = False), row = 3, col = 1)
fig.add_hline(y = 0, row = 3, col = 1, line = dict(color = '#bbbbbb', width = 1))   #negative prices sit below this

fig.update_layout(template = 'plotly_white', height = 820, hovermode = 'x unified',
                  title = dict(text = f'Day-ahead dispatch  -  {date}', x = 0, xanchor = 'left'),
                  legend = dict(orientation = 'h', y = 1.05, x = 0), margin = dict(t = 100, r = 30))
fig.update_xaxes(showspikes = True, spikemode = 'across', spikethickness = 1, spikecolor = '#bbbbbb')
fig.update_xaxes(tickformat = '%H:%M', dtick = 3 * 3600 * 1000, title_text = 'Time of day', row = 3, col = 1)
fig.update_yaxes(title_text = 'Power (MW)', rangemode = 'tozero', row = 1, col = 1)
fig.update_yaxes(title_text = 'SOC (MWh)', row = 2, col = 1)
fig.update_yaxes(title_text = 'Price (EUR/MWh)', row = 3, col = 1)
fig.show()
